# Partie 2 **Fracture Classification Dataset**

Ce Dataset diponible sur kaggle comprend **4083** images dont **3366** images de classe 0 (Sain) et **717** images de classe 1 (Fracture).

Par conséquent, il est important de noter qu'un algorithme se contenant de répondre simplement comment étant sain pour chaque image aura une accuracy de **82,4%**. Cela implique plusieurs changements dans l'entrainement de nos modèles dont la métrique de validation car l'accuracy seul ne sera plus pertinente et l'erreur attribuée aux images comprenant une fracture et etant mal classé devra être plus grande.

## Analyse exploratoire du dataset

### Analyse de la distribution du dataset

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = pd.read_csv('dataset.csv')

In [ ]:
sns.countplot(df, x='fractured' )
plt.show()

In [ ]:
colonnes = ['leg', 'hand', 'hip', 'shoulder', 'mixed', 'hardware', 'multiscan']
df.groupby('fractured')[colonnes].sum().T.plot(kind='bar', figsize=(10, 6), color=['skyblue', 'salmon'])
plt.show()

Lorsqu'une radio est étiquetée avec **hardware = 1**, cela signifie que l'on voit clairement sur l'image :

* Des vis et des plaques d'ostéosynthèse (utilisées pour ressouder les morceaux d'un os cassé).

* Des broches ou des tiges métalliques intra-médullaires (glissées au centre des os longs comme le fémur ou le tibia).

* Des prothèses complètes ou partielles (comme une prothèse de hanche ou d'épaule).

**mixed** correspond au radio qui englobe plusieurs parties du corps. **multiscan** indique si plusieurs radios sur la même photo.

In [ ]:
print(df.isna().sum())
print(f"Il y a {df['multiscan'].sum()} d'images contennants plusieurs radios")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

df_exemples = df[(df['fractured'] == 1) & (df['multiscan']==1)].sample(4)
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Exemples de Radiographies avec Fractures Visibles", fontsize=16, fontweight='bold', y=1.05)
for ax, (index, row) in zip(axes, df_exemples.iterrows()):
    nom_image = row['image_id']
    chemin_complet = os.path.join("images\Fractured", nom_image)
    img = cv2.imread(chemin_complet, cv2.IMREAD_GRAYSCALE)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
plt.show()


On voit bien ici le problème des multiscans...

On s'occupe ici de charger les images, des les redimensionner en 224*224 et on les encode sur un seul canal (niveau de gris).

## Nétoyage du dataset 

In [ ]:
import os
import cv2
import pandas as pd
dossier_principal = "images" 

def rendre_carre_opencv(img):
    hauteur, largeur = img.shape
    max_cote = max(hauteur, largeur)
    
    # Calcul de l'épaisseur des bordures
    haut = (max_cote - hauteur) // 2
    bas = max_cote - hauteur - haut
    gauche = (max_cote - largeur) // 2
    droite = max_cote - largeur - gauche
    
    # Ajout des bordures noires (valeur 0)
    img_carree = cv2.copyMakeBorder(img, haut, bas, gauche, droite, cv2.BORDER_CONSTANT, value=0)
    
    return img_carree


df_multiscan = df[df['multiscan'] == 1]
cpt = 0


for index, row in df_multiscan.iterrows():
    nom_fichier = row['image_id']
    statut_medical = row['fractured']

    if statut_medical == 1:
        nom_sous_dossier = "Fractured"
        chemin_complet = os.path.join(dossier_principal, 'Fractured', nom_fichier)
    else:
        nom_sous_dossier = "Non_fractured"
        chemin_complet = os.path.join(dossier_principal, 'Non_fractured', nom_fichier)
    
    img = cv2.imread(chemin_complet, cv2.IMREAD_GRAYSCALE)
    
    if img is None: print("Manquant:", chemin_complet); continue
       
    hauteur, largeur = img.shape
    milieu_x = largeur // 2
    
    img_gauche = img[:, :milieu_x]
    img_droite = img[:, milieu_x:]
    
    #Création des nouveaux noms de fichiers
    img_gauche = rendre_carre_opencv(img_gauche)
    img_droite = rendre_carre_opencv(img_droite)

    

    nom_base, extension = os.path.splitext(nom_fichier)
    nom_gauche = f"{nom_base}_gauche{extension}"
    nom_droite = f"{nom_base}_droite{extension}"
    
    chemin_gauche = os.path.join(dossier_principal,nom_sous_dossier, nom_gauche)
    chemin_droite = os.path.join(dossier_principal,nom_sous_dossier, nom_droite)
    
    # 5. Sauvegarde des nouvelles images sur le disque dur
   
    cv2.imwrite(chemin_gauche, img_gauche)
    cv2.imwrite(chemin_droite, img_droite)
    
    cpt += 1

print(f" {cpt} images multiscan ont été coupées en deux, créant {cpt * 2} nouvelles images individuelles.")

On vérifie ce qu'a donné le decoupage avec l'ajout de bande noir sur les côtés pour que les images soient bien carrées.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os
import random

images_decoupees = [f for f in os.listdir("images\Fractured") if f.endswith('_gauche.jpg') or f.endswith('_droite.jpg')]
print(len(images_decoupees))
exemples = random.sample(images_decoupees, 4)

# 4. Affichage
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Vérification des Multiscans Découpés et Cadrés (Fractures)", fontsize=16, fontweight='bold', y=1.05)

for ax, nom_image in zip(axes, exemples):
    chemin_complet = os.path.join('images\Fractured', nom_image)
    
    img = cv2.imread(chemin_complet, cv2.IMREAD_GRAYSCALE)
    
    ax.imshow(img, cmap='gray')
    # On ajoute le nom de l'image pour vérifier que c'est bien une _gauche ou _droite
    ax.set_title(nom_image, fontsize=9) 
    ax.axis('off')

plt.tight_layout()
plt.show()

On va maintenant supprimer les images multiscan

In [ ]:
import os
import shutil # Bibliothèque Python pour copier/déplacer des fichiers
import pandas as pd

dossier_corbeille = "multiscan_archives"
os.makedirs(dossier_corbeille, exist_ok=True)

df_multiscan = df[df['multiscan'] == 1]
cpt = 0

for index, row in df_multiscan.iterrows():
    nom_fichier = row['image_id']
    statut_medical = row['fractured']
    if statut_medical == 1:
        chemin_original = os.path.join(dossier_principal, 'Fractured', nom_fichier)
    else:
        chemin_original = os.path.join(dossier_principal, 'Non_fractured', nom_fichier)
    
    if os.path.exists(chemin_original):
        chemin_archive = os.path.join(dossier_corbeille, nom_fichier)
        
        # On déplace l'image sans la supprimer
        shutil.move(chemin_original, chemin_archive)        
        cpt += 1
    else:
        pass # L'image a déjà été déplacée ou supprimée lors d'un test précédent

print(f"{cpt} images originales ont été retirées de l'entraînement.")

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np


dossier_principal = "images" 
transformations = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset_complet = datasets.ImageFolder(root=dossier_principal, transform=transformations)

print(f"Mapping des classes : {dataset_complet.class_to_idx}")
print(f"Nombre total d'images chargées : {len(dataset_complet)}")

dataloader = DataLoader(dataset_complet, batch_size=16, shuffle=True)



## Entrainement sans augentation de données

### 1er model